In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import numpy as np
from pathlib import Path
import cv2
import time

# Env loading
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
os.environ["HF_HOME"] = "./models_cache/SAM_original/"

import sys

sys.path.append("..")
from animgen.core.generated_asset_class import GeneratedAssetClass

from IPython.display import display

In [ ]:
VIEW = 13
MESH_PATH = Path("../generated_data/models/img_mesh_Goldfish.glb")
MODEL_PATH = "facebook/sam3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Generating View
mesh = GeneratedAssetClass(MESH_PATH)

mesh_view = mesh.views[VIEW]
mesh_depth = mesh.depths[VIEW]
render_config = mesh.render_config
view_pose = render_config["camera_poses"][VIEW]
print(view_pose)

In [ ]:
# MODEL SETUP
model = Sam3Model.from_pretrained(MODEL_PATH).to(DEVICE)
processor = Sam3Processor.from_pretrained(MODEL_PATH)

In [ ]:
prompts = ["dorsal fins"]

tic = time.time()
inputs = processor(
    images=mesh_view * len(prompts), text=prompts, return_tensors="pt"
).to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)

result = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs["original_sizes"].tolist(),
)[0]

In [ ]:
print(time.time() - tic)

mask = result["masks"].detach().cpu().numpy()[0]
mask_img = (mask * 255).astype(np.uint8)
depth_img = (mesh_depth / np.max(mesh_depth) * 255).astype(np.uint8)

display(Image.fromarray(mesh_view))
display(Image.fromarray(mask_img))
display(Image.fromarray(depth_img))

In [ ]:
from animgen.renderer.renderer import Renderer, render_multiview
from animgen.utils.camera_position import POSITION_GENERATORS

print(POSITION_GENERATORS.keys())